# Book Spine OCR — Typography Grouping + Google Books Query Pipeline

## Objective

This version intentionally **does NOT classify TITLE / AUTHOR / OTHER**.

Instead it:

1. Detects text from each book spine.
2. Uses OCR polygons and perspective correction.
3. Re-recognizes difficult regions with targeted preprocessing.
4. Removes obvious OCR noise and duplicates.
5. Reconstructs fragmented text lines.
6. Measures **text size, text weight and visual style**.
7. Groups similar text by typography rather than semantic meaning.
8. Joins the typography groups into one comma-separated sentence.
9. Produces a `Google Books Query` suitable for the Google Books Volumes API.

### Example

```text
INTERNATIONAL, THINK & GROW RICH, NAPOLEON HILL
```

The purpose is to send the complete useful text from the spine to Google Books and let the metadata service determine the canonical book title and author.

### Important

- No Tesseract
- No DINOv2 / Transformers
- No KMeans semantic classifier
- No full-image 1x / 1.5x / 2x OCR
- Typography grouping is based on size / weight / visual style / geometry
- The final query preserves visual reading order


# PHASE 1 — Environment

## STEP 1 — Check versions and dependency conflicts

This cell intentionally avoids upgrading the environment.

It only installs **EasyOCR 1.7.2 if EasyOCR is missing** and then runs `pip check`.

The OCR pipeline does not require Tesseract, Transformers or DINOv2.


In [ ]:
# ============================================================
# PHASE 1 / STEP 1
# Environment check
# ============================================================

import sys
import subprocess
import importlib.util
from importlib import metadata

def package_version(name):
    try:
        return metadata.version(name)
    except Exception:
        return "not installed"

# Only install EasyOCR when it is actually missing.
if importlib.util.find_spec("easyocr") is None:
    print("EasyOCR not found. Installing EasyOCR 1.7.2...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "easyocr==1.7.2"
    ])

print("Python       :", sys.version)
print("EasyOCR      :", package_version("easyocr"))
print("PyTorch      :", package_version("torch"))
print("TorchVision  :", package_version("torchvision"))
print("OpenCV       :", package_version("opencv-python-headless"))
print("NumPy        :", package_version("numpy"))
print("Pandas       :", package_version("pandas"))

print("\nRunning pip check...")
check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)

if check.stdout.strip():
    print(check.stdout.strip())
else:
    print("No pip dependency conflicts reported.")

if check.stderr.strip():
    print(check.stderr.strip())


EasyOCR not found. Installing EasyOCR 1.7.2...
Python       : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
EasyOCR      : 1.7.2
PyTorch      : 2.11.0+cu130
TorchVision  : 0.26.0+cu130
OpenCV       : 5.0.0.93
NumPy        : 2.1.3
Pandas       : 2.2.3
Pillow       : 11.3.0
scikit-learn : 1.6.1

Running pip check...
ipython 7.34.0 requires jedi, which is not installed.


## STEP 2 — Imports and GPU check

In [ ]:
# ============================================================
# PHASE 1 / STEP 2
# Imports
# ============================================================

import gc
import re
import math
from pathlib import Path
from difflib import SequenceMatcher

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
import easyocr

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


PyTorch : 2.11.0+cu130
CUDA    : True
GPU     : Tesla T4


# PHASE 2 — Input

## STEP 3 — Configure input/output folders

The notebook expects **already extracted individual book spines**.

Example:

```text
/content/bookImg/
    book_01.jpg
    book_02.jpg
    book_03.jpg
    ...
```

For a real bookshelf application, your segmentation model should create these spine crops first.


In [ ]:
# ============================================================
# PHASE 2 / STEP 3
# Paths and configuration
# ============================================================

INPUT_DIR = Path("/content/bookImg")

OUTPUT_DIR = Path("/content/BookSpine_OCR_Final_Output")
ANNOTATED_DIR = OUTPUT_DIR / "annotated"
NORMALIZED_DIR = OUTPUT_DIR / "normalized_spines"

for d in [
    OUTPUT_DIR,
    ANNOTATED_DIR,
    NORMALIZED_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

extensions = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp",
    ".JPG", ".JPEG", ".PNG", ".BMP", ".WEBP"
}

image_files = sorted(
    p for p in INPUT_DIR.iterdir()
    if p.is_file() and p.suffix in extensions
)

if not image_files:
    raise FileNotFoundError(
        f"No images found in {INPUT_DIR}. "
        "Put individual book-spine images there."
    )

print(f"Found {len(image_files)} spine images.")
for p in image_files:
    print(" -", p.name)


Found 15 spine images.
 - book_01.jpg
 - book_02.jpg
 - book_03.jpg
 - book_04.jpg
 - book_05.jpg
 - book_06.jpg
 - book_07.jpg
 - book_08.jpg
 - book_09.jpg
 - book_10.jpg
 - book_11.jpg
 - book_12.jpg
 - book_13.jpg
 - book_14.jpg
 - book_15.jpg


# PHASE 3 — OCR Engine

## STEP 4 — Load EasyOCR

Only English is enabled because the supplied reference spines are English.

The recognizer uses beam search for better recognition quality.


In [ ]:

# ============================================================
# PHASE 3 / STEP 4
# EasyOCR
# ============================================================

USE_GPU = torch.cuda.is_available()

reader = easyocr.Reader(
    ["en"],
    gpu=USE_GPU,
    verbose=True
)

print("EasyOCR ready.")


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteEasyOCR ready.


# PHASE 4 — Spine Image Normalization

## STEP 5 — Normalize image resolution

Your supplied spine images are often very wide but only a few hundred pixels tall.

The OCR system therefore:

- keeps one normalized scale,
- upscales small spines,
- limits the maximum image size,
- avoids repeatedly running OCR at several full-image scales.

This replaces the old 1x / 1.5x / 2x full-image OCR strategy.


In [ ]:

# ============================================================
# PHASE 4 / STEP 5
# Image normalization
# ============================================================

MAX_LONG_SIDE = 4200
TARGET_HEIGHT = 480

def read_image(path):
    image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)

    if image is None:
        raise ValueError(f"Could not read image: {path}")

    # Handle RGBA images.
    if image.ndim == 3 and image.shape[2] == 4:
        bgr = image[:, :, :3]
        alpha = image[:, :, 3:4].astype(np.float32) / 255.0
        white = np.full_like(bgr, 255)
        image = (
            bgr.astype(np.float32) * alpha
            + white.astype(np.float32) * (1 - alpha)
        ).astype(np.uint8)

    elif image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    return image

def normalize_spine(image):
    h, w = image.shape[:2]

    # Scale small images upward so thin letters have more pixels.
    scale_up = (
        TARGET_HEIGHT / h
        if h < TARGET_HEIGHT
        else 1.0
    )

    # Never exceed maximum long side.
    scale_down = (
        MAX_LONG_SIDE / max(h * scale_up, w * scale_up)
        if max(h * scale_up, w * scale_up) > MAX_LONG_SIDE
        else 1.0
    )

    scale = scale_up * scale_down

    if abs(scale - 1.0) < 0.01:
        return image.copy()

    interpolation = (
        cv2.INTER_CUBIC
        if scale > 1
        else cv2.INTER_AREA
    )

    return cv2.resize(
        image,
        None,
        fx=scale,
        fy=scale,
        interpolation=interpolation
    )

def save_normalized_spine(image, path):
    out = NORMALIZED_DIR / f"{Path(path).stem}_normalized.jpg"
    cv2.imwrite(str(out), image)
    return out


## STEP 6 — Optional safe deskew

Many spines have mild tilt.

This step estimates a dominant foreground angle and corrects only small rotations. Large or uncertain angles are left untouched to avoid damaging the image.


In [ ]:

# ============================================================
# PHASE 4 / STEP 6
# Safe deskew
# ============================================================

def estimate_foreground_angle(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Detect strong edges.
    edges = cv2.Canny(gray, 50, 150)

    # Hough line estimate.
    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 180,
        threshold=max(30, image.shape[1] // 20),
        minLineLength=max(50, image.shape[1] // 10),
        maxLineGap=20
    )

    if lines is None:
        return 0.0

    angles = []

    for line in lines[:, 0]:
        x1, y1, x2, y2 = line

        dx = x2 - x1
        dy = y2 - y1

        if abs(dx) < 1:
            continue

        angle = math.degrees(math.atan2(dy, dx))

        # Only near-horizontal lines are useful here.
        if -15 <= angle <= 15:
            angles.append(angle)

    if not angles:
        return 0.0

    return float(np.median(angles))

def deskew_image(image, max_angle=10.0):
    angle = estimate_foreground_angle(image)

    if abs(angle) < 0.5 or abs(angle) > max_angle:
        return image.copy(), 0.0

    h, w = image.shape[:2]
    center = (w / 2, h / 2)

    matrix = cv2.getRotationMatrix2D(
        center,
        angle,
        1.0
    )

    rotated = cv2.warpAffine(
        image,
        matrix,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(255, 255, 255)
    )

    return rotated, angle


# PHASE 5 — Text Detection

## STEP 7 — Run one high-quality text detection pass

The detector is tuned for:

- long horizontal book-spine text,
- thin characters,
- low-contrast printed text,
- slightly damaged covers.

A fallback detector pass is available only when the primary pass appears to have found too little text.


In [ ]:
# ============================================================
# PHASE 5 / STEP 7
# Text detection
# ============================================================

ALLOWED_CHARS = (
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "abcdefghijklmnopqrstuvwxyz"
    "0123456789"
    ".,!?&'’:-/$*()"
)

DETECT_CONFIG = {
    "text_threshold": 0.40,
    "low_text": 0.20,
    "link_threshold": 0.20,
    "min_size": 6,
    "canvas_size": 3200,
    "mag_ratio": 1.0,
    "slope_ths": 0.15,
    "ycenter_ths": 0.35,
    "height_ths": 0.35,
    "width_ths": 0.50,
    "add_margin": 0.05,
    "contrast_ths": 0.05,
    "adjust_contrast": 0.50,
    "filter_ths": 0.003,
    "decoder": "beamsearch",
    "beamWidth": 5,
    "batch_size": 1,
    "paragraph": False,
    "allowlist": ALLOWED_CHARS,
}

FALLBACK_DETECT_CONFIG = {
    **DETECT_CONFIG,
    "text_threshold": 0.30,
    "low_text": 0.12,
    "link_threshold": 0.15,
    "height_ths": 0.28,
    "width_ths": 0.45,
}

def run_easyocr_detection(image, config):
    rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    return reader.readtext(
        rgb,
        detail=1,
        **config
    )

def detections_to_regions(detections):
    regions = []

    for det in detections:
        if len(det) != 3:
            continue

        polygon, text, confidence = det

        text = clean_ocr_text(text)
        confidence = float(confidence)

        if not valid_ocr_text(text):
            continue

        polygon = np.asarray(
            polygon,
            dtype=np.float32
        )

        regions.append({
            "polygon": polygon.tolist(),
            "box": polygon_to_bbox(polygon),
            "text": text,
            "confidence": confidence
        })

    return regions


# PHASE 6 — Region Quality Filtering

## STEP 8 — Remove obvious OCR noise

The goal is **not** to discard small legitimate author text.

Instead, only remove high-confidence noise patterns such as:

- ISBN numbers,
- long numeric strings,
- barcode-like OCR,
- URLs,
- isolated one-character regions,
- extremely tiny low-confidence detections.

Publisher names and logos are not always removed here; they are usually classified as `OTHER`, preserving information for debugging.


In [ ]:
# ============================================================
# PHASE 6 / STEP 8
# Cleaning and duplicate removal
# ============================================================

MIN_REGION_HEIGHT_RATIO = 0.010
MIN_ALNUM_CHARACTERS = 2

ISBN_PATTERNS = [
    re.compile(r"^(?:ISBN[:\s-]*)?97[89][0-9Xx -]{9,}$"),
    re.compile(r"^[0-9Xx -]{10,}$"),
]

NOISE_WORDS = {
    "WWW",
    "HTTP",
    "HTTPS",
    "COM",
    "ORG",
    "NET",
}

def clean_ocr_text(text):
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip(" |[]{}").strip()

def valid_ocr_text(text):
    text = clean_ocr_text(text)

    if len(text) < 2:
        return False

    alnum_count = sum(
        c.isalnum()
        for c in text
    )

    return alnum_count >= MIN_ALNUM_CHARACTERS

def polygon_to_bbox(polygon):
    pts = np.asarray(
        polygon,
        dtype=np.float32
    )

    return [
        float(pts[:, 0].min()),
        float(pts[:, 1].min()),
        float(pts[:, 0].max()),
        float(pts[:, 1].max())
    ]

def bbox_area(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2-x1) * max(0.0, y2-y1)

def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    inter = (
        max(0.0, ix2-ix1)
        * max(0.0, iy2-iy1)
    )

    union = (
        bbox_area(a)
        + bbox_area(b)
        - inter
    )

    return inter / union if union else 0.0

def bbox_area_ratio(a, b):
    aa = bbox_area(a)
    ab = bbox_area(b)

    if max(aa, ab) == 0:
        return 0.0

    return min(aa, ab) / max(aa, ab)

def normalized_for_comparison(text):
    text = clean_ocr_text(text).upper()
    text = re.sub(
        r"[^A-Z0-9 ]+",
        " ",
        text
    )
    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()

def text_similarity(a, b):
    a = normalized_for_comparison(a)
    b = normalized_for_comparison(b)

    if not a or not b:
        return 0.0

    return SequenceMatcher(
        None,
        a,
        b
    ).ratio()

def looks_like_isbn(text):
    compact = re.sub(
        r"[\s-]+",
        "",
        text
    )

    return any(
        p.fullmatch(compact)
        for p in ISBN_PATTERNS
    )

def looks_like_url_or_noise(text):
    upper = text.upper()

    if any(
        token in upper
        for token in NOISE_WORDS
    ):
        return True

    if re.search(
        r"(https?://|www\.)",
        text,
        flags=re.IGNORECASE
    ):
        return True

    return False

def filter_regions(regions, image_shape):
    h_img = image_shape[0]

    filtered = []

    for region in regions:
        text = clean_ocr_text(region["text"])

        height = (
            region["box"][3]
            - region["box"][1]
        )

        height_ratio = (
            height / max(1, h_img)
        )

        # Very tiny boxes are usually logos/noise.
        # Keep somewhat larger small text because author lines
        # can legitimately be small.
        if (
            height_ratio < MIN_REGION_HEIGHT_RATIO
            and region["confidence"] < 0.70
        ):
            continue

        if looks_like_isbn(text):
            continue

        if looks_like_url_or_noise(text):
            continue

        region["height_ratio"] = float(height_ratio)
        filtered.append(region)

    return filtered

def deduplicate_regions(regions):
    # Highest confidence first.
    regions = sorted(
        regions,
        key=lambda r: (
            r["confidence"],
            r.get("height_ratio", 0.0)
        ),
        reverse=True
    )

    kept = []

    for candidate in regions:
        duplicate = False

        for existing in kept:

            iou = bbox_iou(
                candidate["box"],
                existing["box"]
            )

            area_ratio = bbox_area_ratio(
                candidate["box"],
                existing["box"]
            )

            text_sim = text_similarity(
                candidate["text"],
                existing["text"]
            )

            # Same region with high geometry overlap.
            if (
                iou >= 0.55
                and area_ratio >= 0.55
            ):
                duplicate = True
                break

            # Same text repeated in almost the same location.
            if (
                text_sim >= 0.90
                and iou >= 0.35
            ):
                duplicate = True
                break

        if not duplicate:
            kept.append(candidate)

    return kept


## STEP 9 — Adaptive detector fallback

We first trust the primary detector.

Only when it appears to have found very little useful text do we run **one fallback pass**.

This avoids the duplicate explosion from the old multi-scale pipeline.


In [ ]:

# ============================================================
# PHASE 6 / STEP 9
# Adaptive detection
# ============================================================

def adaptive_detect(image):
    primary_raw = run_easyocr_detection(
        image,
        DETECT_CONFIG
    )

    primary = detections_to_regions(
        primary_raw
    )

    primary = filter_regions(
        primary,
        image.shape
    )

    # Trigger fallback only when primary is clearly weak.
    primary_total_chars = sum(
        len(r["text"])
        for r in primary
    )

    should_fallback = (
        len(primary) == 0
        or (
            len(primary) <= 1
            and primary_total_chars < 8
        )
    )

    if not should_fallback:
        return deduplicate_regions(primary), "primary"

    fallback_raw = run_easyocr_detection(
        image,
        FALLBACK_DETECT_CONFIG
    )

    fallback = detections_to_regions(
        fallback_raw
    )

    fallback = filter_regions(
        fallback,
        image.shape
    )

    combined = deduplicate_regions(
        primary + fallback
    )

    return combined, "primary+fallback"


# PHASE 7 — Polygon Text Crops

## STEP 10 — Perspective-correct every OCR region

This is critical.

The OCR engine returns quadrilateral polygons. The polygon is transformed to a clean horizontal rectangle before recognition.

This prevents perspective distortion from weakening OCR.


In [ ]:
# ============================================================
# PHASE 7 / STEP 10
# Polygon perspective crop
# ============================================================

TEXT_CROP_MIN_HEIGHT = 48
TEXT_CROP_MAX_HEIGHT = 180

def order_quad_points(points):
    pts = np.asarray(
        points,
        dtype=np.float32
    )

    if pts.shape != (4, 2):
        return None

    s = pts.sum(axis=1)
    d = np.diff(
        pts,
        axis=1
    ).ravel()

    rect = np.zeros(
        (4, 2),
        dtype=np.float32
    )

    rect[0] = pts[np.argmin(s)]  # TL
    rect[2] = pts[np.argmax(s)]  # BR
    rect[1] = pts[np.argmin(d)]  # TR
    rect[3] = pts[np.argmax(d)]  # BL

    return rect

def perspective_text_crop(
    image,
    polygon,
    padding=8
):
    pts = np.asarray(
        polygon,
        dtype=np.float32
    )

    if pts.shape != (4, 2):
        x1,y1,x2,y2 = polygon_to_bbox(pts)

        x1 = max(
            0,
            int(round(x1)) - padding
        )
        y1 = max(
            0,
            int(round(y1)) - padding
        )
        x2 = min(
            image.shape[1],
            int(round(x2)) + padding
        )
        y2 = min(
            image.shape[0],
            int(round(y2)) + padding
        )

        return (
            image[y1:y2, x1:x2].copy()
            if x2 > x1 and y2 > y1
            else None
        )

    rect = order_quad_points(pts)

    if rect is None:
        return None

    tl, tr, br, bl = rect

    def dist(a, b):
        return float(
            np.linalg.norm(a-b)
        )

    width = max(
        dist(tl, tr),
        dist(bl, br)
    )

    height = max(
        dist(tl, bl),
        dist(tr, br)
    )

    width = int(round(width))
    height = int(round(height))

    if width < 3 or height < 3:
        return None

    dst_w = width + 2 * padding
    dst_h = height + 2 * padding

    dst = np.array([
        [padding, padding],
        [dst_w-padding-1, padding],
        [dst_w-padding-1, dst_h-padding-1],
        [padding, dst_h-padding-1]
    ], dtype=np.float32)

    matrix = cv2.getPerspectiveTransform(
        rect,
        dst
    )

    crop = cv2.warpPerspective(
        image,
        matrix,
        (dst_w, dst_h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(255,255,255)
    )

    return crop if crop.size else None

def resize_text_crop_for_recognition(crop):
    h = crop.shape[0]

    if h < TEXT_CROP_MIN_HEIGHT:
        scale = TEXT_CROP_MIN_HEIGHT / max(1, h)

    elif h > TEXT_CROP_MAX_HEIGHT:
        scale = TEXT_CROP_MAX_HEIGHT / h

    else:
        scale = 1.0

    if abs(scale-1.0) < 0.01:
        return crop

    return cv2.resize(
        crop,
        None,
        fx=scale,
        fy=scale,
        interpolation=(
            cv2.INTER_CUBIC
            if scale > 1
            else cv2.INTER_AREA
        )
    )


# PHASE 8 — Recognition Improvement

## STEP 11 — Selective preprocessing

Preprocessing is applied **only to individual difficult text crops**.

Variants:

- Original
- CLAHE-enhanced grayscale
- Otsu threshold

We do not run these expensive variants over the entire spine.


In [ ]:

# ============================================================
# PHASE 8 / STEP 11
# Recognition variants
# ============================================================

RECOGNITION_CONFIDENCE_THRESHOLD = 0.72

def recognition_variants(crop):
    variants = [
        ("original", crop)
    ]

    gray = cv2.cvtColor(
        crop,
        cv2.COLOR_BGR2GRAY
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    enhanced = clahe.apply(gray)

    variants.append((
        "clahe",
        cv2.cvtColor(
            enhanced,
            cv2.COLOR_GRAY2BGR
        )
    ))

    _, otsu = cv2.threshold(
        enhanced,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    variants.append((
        "otsu",
        cv2.cvtColor(
            otsu,
            cv2.COLOR_GRAY2BGR
        )
    ))

    return variants

def recognize_text_crop(crop):
    results = []

    for variant_name, variant in recognition_variants(crop):

        rgb = cv2.cvtColor(
            variant,
            cv2.COLOR_BGR2RGB
        )

        try:
            detections = reader.readtext(
                rgb,
                detail=1,
                paragraph=False,

                decoder="beamsearch",
                beamWidth=5,

                allowlist=ALLOWED_CHARS,

                contrast_ths=0.05,
                adjust_contrast=0.50,
                filter_ths=0.003,

                text_threshold=0.35,
                low_text=0.15,
                link_threshold=0.15,

                canvas_size=2600,
                mag_ratio=1.15,

                batch_size=1
            )
        except Exception:
            continue

        if not detections:
            continue

        best = max(
            detections,
            key=lambda x: float(x[2])
        )

        text = clean_ocr_text(best[1])
        confidence = float(best[2])

        if valid_ocr_text(text):
            results.append({
                "variant": variant_name,
                "text": text,
                "confidence": confidence
            })

    if not results:
        return {
            "text": "",
            "confidence": 0.0,
            "variant": None,
            "agreement": 0
        }

    best = None
    best_score = -1.0
    best_agreement = 0

    for candidate in results:

        agreement = sum(
            text_similarity(
                candidate["text"],
                other["text"]
            ) >= 0.72
            for other in results
            if other is not candidate
        )

        # Confidence plus agreement.
        score = (
            candidate["confidence"]
            + 0.08 * agreement
        )

        if score > best_score:
            best_score = score
            best = candidate
            best_agreement = agreement

    return {
        "text": best["text"],
        "confidence": float(best["confidence"]),
        "variant": best["variant"],
        "agreement": int(best_agreement)
    }


## STEP 12 — Improve recognition of low-confidence regions

High-confidence EasyOCR detections are kept.

Only low-confidence regions are re-recognized on the polygon crop.


In [ ]:
# ============================================================
# PHASE 8 / STEP 12
# Region recognition
# ============================================================

def refine_region_recognition(
    image,
    region
):
    # Keep strong detections without unnecessary cropping.
    if region["confidence"] >= RECOGNITION_CONFIDENCE_THRESHOLD:
        return region

    crop = perspective_text_crop(
        image,
        region["polygon"],
        padding=8
    )

    if crop is None:
        return region

    crop = resize_text_crop_for_recognition(
        crop
    )

    retry = recognize_text_crop(
        crop
    )

    if (
        retry["text"]
        and retry["confidence"]
        >= region["confidence"]
    ):
        region["text"] = retry["text"]
        region["confidence"] = retry["confidence"]

    return region
